# Stage 11 — Leakage-Safe + Best-vs-Last Sensitivity

This notebook addresses two external-review concerns in one frozen analysis:

1. cross-split same-scene / near-duplicate sensitivity;
2. final-epoch `last.pt` versus validation-selected `best.pt`.

**No training is performed.** All 24 cells are evaluation-only on the same leakage-safe test subsets.

In [ ]:
# Cell 1 — Mount Drive and verify GPU
from pathlib import Path
from google.colab import drive
import shutil, subprocess

drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
RUN_ROOT = DRIVE / 'Stage5_Baselines'
OUT = DRIVE / 'Stage11_LeakageSafe_Sensitivity'
OUT.mkdir(parents=True, exist_ok=True)

if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU runtime required. Choose Runtime > Change runtime type > T4 GPU.')
subprocess.run(['nvidia-smi'], check=True)
print('STAGE11_GPU_PREFLIGHT: PASS')

In [ ]:
# Cell 2 — Reassemble the exact frozen Stage 4.5 h9-v2 bundle
import hashlib, zipfile, shutil

PARTS = [
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02',
]
missing=[str(p) for p in PARTS if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing frozen bundle parts: '+', '.join(missing))

BUNDLE=Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED_SHA='dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h=hashlib.sha256()
with BUNDLE.open('wb') as out:
    for p in PARTS:
        with p.open('rb') as f:
            for ch in iter(lambda:f.read(8*1024*1024),b''):
                out.write(ch); h.update(ch)
assert h.hexdigest()==EXPECTED_SHA

DATA=Path('/content/stage11_frozen_data')
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert z.testzip() is None
    z.extractall(DATA)
Y_DATA=DATA/'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA=DATA/'h9_v2/Construction-PPE-h9-v2/data.yaml'
assert Y_DATA.is_file() and C_DATA.is_file()
print('STAGE11_FROZEN_DATA: PASS')

In [ ]:
# Cell 3 — Clone reviewer-remediation branch and install exact evaluator
import subprocess, shutil, sys

REPO=Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--branch','stage11-reviewer-remediation','--single-branch',
    'https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git',str(REPO)
],check=True)
subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'ultralytics==8.4.156','PyYAML==6.0.3','pandas'
],check=True)

import ultralytics, torch
assert ultralytics.__version__=='8.4.156'
assert torch.cuda.is_available()
print('STAGE11_CODE_ENV: PASS')

In [ ]:
# Cell 4 — Execute 24 leakage-safe evaluation cells
import subprocess, sys

SCRIPT=REPO/'cross_domain_reliability/stage11_reviewer_remediation/evaluate_leakage_safe_sensitivity.py'
EXCL=REPO/'cross_domain_reliability/stage11_reviewer_remediation/high_confidence_test_exclusions.json'

subprocess.run([
    sys.executable,str(SCRIPT),
    '--stage5-root',str(RUN_ROOT),
    '--y-data',str(Y_DATA),
    '--c-data',str(C_DATA),
    '--exclusions',str(EXCL),
    '--out',str(OUT),
    '--device','0',
    '--batch','8',
    '--imgsz','640',
    '--ap-conf','0.001',
    '--operating-conf','0.25',
    '--nms-iou','0.70',
    '--match-iou','0.50',
],check=True)

In [ ]:
# Cell 5 — Display the decision tables
import pandas as pd, json
from IPython.display import display

print('Group summary')
display(pd.read_csv(OUT/'stage11_group_summary.csv'))

print('\nBest vs last paired sensitivity')
display(pd.read_csv(OUT/'stage11_best_vs_last_paired.csv'))

print('\nConsensus false-negative summary')
display(pd.read_csv(OUT/'stage11_consensus_fn_summary.csv'))

print('\nValidation checkpoint sensitivity')
display(pd.read_csv(OUT/'stage11_validation_checkpoint_sensitivity.csv'))

status=json.loads((OUT/'stage11_status.json').read_text())
print('\n',json.dumps(status,indent=2))
print('STAGE11_COMPLETE')